# Chasher-Disha — YOLOv8-nano Training (Google Colab)

**M4 / ML-1** — RiceLeafDiseaseBD, 6 classes

**Runtime:** GPU (T4 recommended)  
**Estimated time:** 2–4 hours for 100 epochs on a T4  

## Instructions
1. **Runtime → Change runtime type → T4 GPU**
2. Upload `RiceLeafDiseaseBD.zip` to your Google Drive (`/MyDrive/datasets/`)
3. Run all cells in order
4. Download `best.pt` and `best.onnx` from the output cell

In [ ]:
# Cell 1 — Install dependencies
!pip install ultralytics>=8.3.0 split-folders --quiet

In [ ]:
# Cell 2 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 3 — Extract dataset
import zipfile, os
ZIP_PATH    = '/content/drive/MyDrive/datasets/RiceLeafDiseaseBD.zip'
EXTRACT_DIR = '/content/RiceLeafDiseaseBD'

if not os.path.exists(EXTRACT_DIR):
    print('Extracting…')
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(EXTRACT_DIR)
    print('Done.')
else:
    print('Already extracted.')

In [ ]:
# Cell 4 — Inspect dataset structure
import os
for root, dirs, files in os.walk('/content/RiceLeafDiseaseBD'):
    level = root.replace('/content/RiceLeafDiseaseBD', '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/ ({len(files)} files)')
    if level > 2:
        break

In [ ]:
# Cell 5 — Write data.yaml
# Adjust train/val/test paths if the dataset has a different layout
DATA_YAML = '''
path: /content/RiceLeafDiseaseBD
train: images/train
val:   images/val
test:  images/test

nc: 6
names:
  - Healthy
  - Blast
  - Brown Spot
  - Leaf Smut
  - Rice Tungro
  - Sheath Blight
'''
with open('/content/data.yaml', 'w') as f:
    f.write(DATA_YAML.strip())
print('data.yaml written.')

In [ ]:
# Cell 6 — Train YOLOv8-nano
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
results = model.train(
    data='/content/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    device='cuda',
    project='/content/runs/train',
    name='rice_disease_v1',
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    flipud=0.3, fliplr=0.5, mosaic=1.0, mixup=0.1,
)
metrics = results.results_dict
print('mAP@0.5      :', metrics.get('metrics/mAP50(B)'))
print('mAP@0.5:0.95 :', metrics.get('metrics/mAP50-95(B)'))

In [ ]:
# Cell 7 — Export to ONNX (for Capacitor/WASM inference)
model_best = YOLO('/content/runs/train/rice_disease_v1/weights/best.pt')
model_best.export(format='onnx', imgsz=640, simplify=True)
print('ONNX export complete.')

In [ ]:
# Cell 8 — Copy weights to Drive for download
import shutil, os
OUT_DIR = '/content/drive/MyDrive/chasher-disha-model'
os.makedirs(OUT_DIR, exist_ok=True)
shutil.copy2('/content/runs/train/rice_disease_v1/weights/best.pt',
             f'{OUT_DIR}/rice_disease.pt')
onnx = '/content/runs/train/rice_disease_v1/weights/best.onnx'
if os.path.exists(onnx):
    shutil.copy2(onnx, f'{OUT_DIR}/rice_disease.onnx')
print(f'Weights saved to Google Drive: {OUT_DIR}')

## Next steps

1. Download `rice_disease.pt` and `rice_disease.onnx` from Google Drive.
2. Place them at:
   ```
   chatbot_app/ml/models/rice_disease.pt
   chatbot_app/ml/models/rice_disease.onnx
   ```
3. Run `python manage.py runserver` — the disease endpoint is now live.